# Phase 6 (post hoc): the full grid with a patient-baseline RR ratio (phase6_rr_baseline)

PREREGISTRATION §10 v5, P5. Rebuilds dataset v2 from the frozen v1 beats and the DS1 annotations (only `rr_ratio`
changes: `rr_prev` over the median RR within ±150 s), remaps the RR ratio of the frozen phase 3 banks, checks both
against the hashes in `results/phase6_rr_baseline/data.json`, then runs the phase 4a gate's full grid unchanged on v2
(96 runs, resumable). Validation data only: selection and the DS2 part run locally afterwards.

1. Settings: **Internet on**. Accelerator: **GPU**. Add the datasets `mitdb-raw` and `ecgaug-banks` (the frozen banks).
   Save Version > Save & Run All.
2. Add-ons > Secrets: `GITHUB_TOKEN` (private repo), ticked for this notebook.
3. Download the zip in Output and run `python kaggle/ingest_runs.py <zip>`; then `--part select`, `--part references`,
   `--part test` (with `ALLOW_TEST_POSTHOC=1`) and `--part external` locally.


In [ ]:
# --- Parameters: edit only this cell ---------------------------------------------------------
BANKS_DATASET_DIR = "/kaggle/input/ecgaug-banks"   # the attached Kaggle dataset with the frozen phase 3 banks
SMOKE_TEST = False                    # kept for the notebook layout; this experiment has no smoke mode (its unit tests run locally)
GIT_REF = "main"                      # branch name or full commit SHA to run
DEVICE = "auto"                       # "auto" uses the GPU when the session has one
ENV_MODE = "locked"                   # "locked" = exact uv.lock versions; "kaggle" = preinstalled packages (faster, debugging only)
RAW_DATASET_DIR = "/kaggle/input"     # attached mitdb-raw files; if none, downloaded from PhysioNet's AWS mirror (checksums checked)
REPO_URL = "https://github.com/akashsaha477/Gen_Ai_Project.git"
PUSH_RESULTS = False                  # True = commit and push the results to GIT_REF from Kaggle; default: ingest the zip yourself
GIT_NAME = ""                         # your name for the results commit, e.g. "Akash Saha"
GIT_EMAIL = ""                        # your GitHub email for the results commit


In [ ]:
import os, subprocess, sys
from pathlib import Path

WORK = Path("/kaggle/working")
REPO = WORK / "repo"
env = dict(os.environ, GIT_TERMINAL_PROMPT="0")
url = REPO_URL
try:
    from kaggle_secrets import UserSecretsClient
    token = UserSecretsClient().get_secret("GITHUB_TOKEN")
except Exception:
    token = None                      # public repository, or the secret is not attached
if token:
    askpass = WORK / ".git-askpass.sh"
    askpass.write_text('#!/bin/sh\necho "$GITHUB_TOKEN"\n')
    askpass.chmod(0o700)
    env.update(GIT_ASKPASS=str(askpass), GITHUB_TOKEN=token)   # the token is never printed or put in a URL
    url = REPO_URL.replace("https://", "https://x-access-token@")

if not REPO.exists():
    subprocess.run(["git", "clone", "--quiet", url, str(REPO)], check=True, env=env)
subprocess.run(["git", "fetch", "--quiet", "origin"], cwd=REPO, check=True, env=env)
ref = GIT_REF if len(GIT_REF) == 40 else f"origin/{GIT_REF}"
subprocess.run(["git", "checkout", "--quiet", "--detach", ref], cwd=REPO, check=True)
sha = subprocess.run(["git", "rev-parse", "HEAD"], cwd=REPO, capture_output=True, text=True).stdout.strip()
print("running commit", sha)

In [ ]:
configs = ["configs/phase6_rr_baseline/rr_baseline.yaml"]
extra = ["--part", "grid", "--device", DEVICE]
cmd = [sys.executable, "kaggle/kaggle_run.py", "--cli", "ecgaug.experiments.phase6_rr_baseline", "--env", ENV_MODE, "--out", str(WORK),
       "--config", *configs]
cmd += ["--raw-dir", RAW_DATASET_DIR]   # the MIT-BIH files are copied from here and checked against the committed checksums
if BANKS_DATASET_DIR:
    cmd += ["--artifacts-dir", BANKS_DATASET_DIR]
if PUSH_RESULTS:
    branch = GIT_REF if len(GIT_REF) != 40 else "kaggle-results"
    cmd += ["--push-to", branch, "--git-name", GIT_NAME, "--git-email", GIT_EMAIL]
cmd += ["--", *extra]
code = subprocess.run(cmd, cwd=REPO, env=env).returncode   # env carries the token for the push
print("finished with exit code", code, "(2 = the run was refused, read the message above)")
